# PSGC Bronze Ingestion

This notebook ingests the PSA **Philippine Standard Geographic Code (PSGC), 2Q 2026** Excel files from Google Drive into Bronze Delta tables.

| Dataset | File |
| --- | --- |
| `DS_33` | `DS_33_PSA_PSGC_2Q_2026_National_and_Provincial_Summary_2026.xlsx` |
| `DS_34` | `DS_34_PSA_PSGC_2Q_2026_Publication_Datafile_2026.xlsx` |

Each workbook is read directly from Google Drive through the existing Unity Catalog connection. Every sheet is loaded **exactly as printed** (all values kept as text), reconciliation checks are run, and results are logged to the shared ingestion control tables created by `00_ingestion_setup`.

**Bronze outputs** (a rerun replaces only the selected datasets; others are untouched):

| Table | One row = |
| --- | --- |
| `psa_psgc_ds_33_<sheet>` / `psa_psgc_ds_34_<sheet>` (one table per sheet) | one printed worksheet row, with `source_row_number` |
| `psa_psgc_source_files` (all datasets) | one original Excel file stored as binary, for traceability |

No geography mapping, deduplication, type conversion, or missing-value handling happens here. That belongs in Silver.

In [0]:
%pip install openpyxl==3.1.5

In [0]:
from pyspark.sql import functions as F
import uuid, io, re
from datetime import datetime

# ── Google Drive connection (existing Unity Catalog connection) ──
GOOGLE_DRIVE_CONNECTION = "capstone_gdrive"

# ── Unity Catalog targets ────────────────────────────────────────
CATALOG = "tuklas_dev"
BRONZE_SCHEMA = "02_bronze"
INGESTION_SCHEMA = "01_ingestion"
NOTEBOOK_NAME = "02_ingest_psgc"
SOURCE_SYSTEM = "PSA"

run_id = f"psgc_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# ── Google Drive source folder ───────────────────────────────────
# Listed below to find each file's Drive path by exact filename.
PSGC_FOLDER = "https://drive.google.com/drive/folders/1Xtf804IXEblOVhbxeZitGgL-UjPy8AsW"

# ── One entry per workbook. "url" is optional: leave None to use the path found in the folder listing. ──
SOURCE_FILES = {
    "DS_33": {"file": "DS_33_PSA_PSGC_2Q_2026_National_and_Provincial_Summary_2026.xlsx",
              "table_prefix": "psa_psgc_ds_33", "url": None},
    "DS_34": {"file": "DS_34_PSA_PSGC_2Q_2026_Publication_Datafile_2026.xlsx",
              "table_prefix": "psa_psgc_ds_34", "url": None},
}

# Datasets to process in this run: "all", or a list like ["DS_33"] for the pilot
FILES_TO_RUN = "all"

# Header row (1 = first row of the sheet). Override per sheet if a sheet has title rows above its header:
#   HEADER_ROW_OVERRIDES = {("DS_33", "Summary"): 3}
DEFAULT_HEADER_ROW = 1
HEADER_ROW_OVERRIDES = {}

# ── Bronze tables: one data table per sheet + one combined table for the original files ──
TBL_FILES = "psa_psgc_source_files"

def bronze_table(name): return f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{name}`"
def ingestion_table(name): return f"`{CATALOG}`.`{INGESTION_SCHEMA}`.`{name}`"

def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_") or "sheet"

def tbl_sheet(dataset_id, sheet_name): return f"{SOURCE_FILES[dataset_id]['table_prefix']}_{slug(sheet_name)}"

METADATA_COLS = ["_source_dataset_id", "_source_file", "_source_sheet", "_source_path", "_source_system", "_ingested_at"]

def add_bronze_metadata(df, dataset_id, source_file, source_sheet, source_path):
    return (df
        .withColumn("_source_dataset_id", F.lit(dataset_id))
        .withColumn("_source_file", F.lit(source_file))
        .withColumn("_source_sheet", F.lit(source_sheet).cast("string"))
        .withColumn("_source_path", F.lit(source_path))
        .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("_ingested_at", F.current_timestamp()))

ingestion_summary = []

def record_ingestion(dataset_id, source_file, source_sheet, bronze_tbl, row_count, col_count, status, notes="", source_path=None):
    """Append to the in-memory summary and MERGE one row into file_manifest (same pattern as 01_ingest_psa)."""
    ingestion_summary.append({"source_dataset_id": dataset_id, "source_file": source_file, "source_sheet": source_sheet,
                              "bronze_table": bronze_tbl, "row_count": row_count, "column_count": col_count,
                              "status": status, "notes": notes})
    row = spark.createDataFrame([(run_id, NOTEBOOK_NAME, dataset_id, SOURCE_SYSTEM, source_file, source_sheet, source_path,
                                  bronze_tbl, row_count, col_count, status, notes)],
        schema="run_id STRING, notebook_name STRING, source_dataset_id STRING, source_system STRING, source_file STRING, "
               "source_sheet STRING, source_path STRING, destination_table STRING, row_count BIGINT, column_count BIGINT, "
               "status STRING, notes STRING")
    row = row.withColumn("discovered_at", F.current_timestamp()).withColumn("ingested_at", F.current_timestamp())
    row.createOrReplaceTempView("_manifest_row")
    spark.sql(f"""
    MERGE INTO {ingestion_table('file_manifest')} AS target
    USING _manifest_row AS source
    ON target.run_id = source.run_id
       AND target.source_dataset_id = source.source_dataset_id
       AND target.source_file = source.source_file
       AND COALESCE(target.source_sheet, '') = COALESCE(source.source_sheet, '')
       AND target.destination_table = source.destination_table
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)

def log_ingestion_error(dataset_id, source_file, source_sheet, source_path, destination_table, error_type, error_message, notes=""):
    row = spark.createDataFrame([(run_id, NOTEBOOK_NAME, dataset_id, SOURCE_SYSTEM, source_file, source_sheet, source_path,
                                  destination_table, error_type, str(error_message)[:4000], None, notes)],
        schema="run_id STRING, notebook_name STRING, source_dataset_id STRING, source_system STRING, source_file STRING, "
               "source_sheet STRING, source_path STRING, destination_table STRING, error_type STRING, "
               "error_message STRING, error_timestamp TIMESTAMP, notes STRING")
    row = row.withColumn("error_timestamp", F.current_timestamp())
    row.write.format("delta").mode("append").saveAsTable(ingestion_table("ingestion_errors"))

def write_table(df, tbl):
    """Each sheet has its own table, so a rerun simply overwrites it (no duplicates)."""
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(bronze_table(tbl)))

def write_dataset_part(df, tbl, dataset_id):
    """Combined table: replace only this dataset's rows; other datasets are left untouched."""
    if spark.catalog.tableExists(bronze_table(tbl)):
        (df.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"_source_dataset_id = '{dataset_id}'")
           .saveAsTable(bronze_table(tbl)))
    else:
        df.write.format("delta").mode("overwrite").saveAsTable(bronze_table(tbl))

datasets = list(SOURCE_FILES) if FILES_TO_RUN == "all" else FILES_TO_RUN
print(f"Run ID  : {run_id}\nDatasets: {datasets}")

## Excel Extraction Functions

Each worksheet is read cell by cell with `openpyxl` and kept **exactly as printed** (all values as text).

* **Header row:** taken from `DEFAULT_HEADER_ROW` (or a per-sheet override). Header names are converted to safe snake_case column names; blank or duplicate headers get a positional name so no column is lost.
* **Row traceability:** every data row keeps its worksheet row number as `source_row_number`.
* **Text values:** whole-number cells are written without a trailing `.0` (so `1300000000` stays `1300000000`). Codes that were saved as numbers in Excel have already lost any leading zeros in the file itself, so Silver should re-pad codes if needed.
* **Empty rows:** rows with no values at all are dropped; nothing else is removed.

Checks run per sheet:

* the sheet has a header and at least one data row
* blank header cells are given positional names and recorded in notes (informational)
* rows written to Bronze = data rows read from the sheet

In [0]:
from openpyxl import load_workbook

def _cell_text(v):
    """Value exactly as printed, as text. None stays None."""
    if v is None:
        return None
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    s = str(v).strip() if isinstance(v, str) else str(v)
    return s if s != "" else None

def _clean_headers(raw_headers):
    """snake_case, never blank, never duplicated."""
    out, seen = [], {}
    for i, h in enumerate(raw_headers, start=1):
        name = slug(h) if h is not None else f"col_{i}"
        if name[0].isdigit():
            name = f"c_{name}"
        if name in ("source_row_number",):
            name = f"{name}_src"
        seen[name] = seen.get(name, 0) + 1
        out.append(name if seen[name] == 1 else f"{name}_{seen[name]}")
    return out

def parse_excel(xlsx_bytes, dataset_id):
    """Extract every worksheet of an Excel workbook.

    Returns a list of dicts, one per sheet:
      {sheet, header_row, columns, raw_headers, rows(list of tuples incl. source_row_number), notes}
    All values are text, exactly as printed.
    """
    wb = load_workbook(io.BytesIO(xlsx_bytes), read_only=True, data_only=True)
    sheets = []
    for ws in wb.worksheets:
        header_row = HEADER_ROW_OVERRIDES.get((dataset_id, ws.title), DEFAULT_HEADER_ROW)
        all_rows = [(i, [_cell_text(c) for c in row]) for i, row in enumerate(ws.iter_rows(values_only=True), start=1)]
        all_rows = [(i, r) for i, r in all_rows if any(v is not None for v in r)]
        header = next((r for i, r in all_rows if i == header_row), None)
        data = [(i, r) for i, r in all_rows if i > header_row]
        width = max([len(header) if header else 0] + [len(r) for _, r in data]) if (header or data) else 0
        raw_headers = (header or [])
        raw_headers = raw_headers + [None] * (width - len(raw_headers))
        # trim trailing columns that are blank in the header AND in every data row
        used = [j for j in range(width)
                if raw_headers[j] is not None or any(j < len(r) and r[j] is not None for _, r in data)]
        last = (max(used) + 1) if used else 0
        raw_headers = raw_headers[:last]
        rows = [tuple([str(i)] + [(r[j] if j < len(r) else None) for j in range(last)]) for i, r in data]
        sheets.append({"sheet": ws.title, "header_row": header_row, "raw_headers": raw_headers,
                       "columns": ["source_row_number"] + _clean_headers(raw_headers), "rows": rows, "header_found": header is not None})
    wb.close()
    return sheets

def check_sheet(sh):
    """Reconciliation checks. Returns (critical_errors, notes)."""
    errors, notes = [], []
    if not sh["header_found"]:
        errors.append(f"No header found at row {sh['header_row']}")
    if len(sh["rows"]) == 0:
        errors.append("No data rows found")
    blank_headers = sum(1 for h in sh["raw_headers"] if h is None)
    if blank_headers:
        notes.append(f"{blank_headers} blank header(s) given positional names")
    return errors, notes

In [0]:
spark.sql(f"""
INSERT INTO {ingestion_table('ingestion_run_log')}
(run_id, notebook_name, source_system, source_folder, started_at, completed_at,
 files_discovered, files_ingested, files_skipped, files_failed, tables_created, status, notes)
VALUES (:run_id, :nb, :sys, :folder, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME, "sys": SOURCE_SYSTEM, "folder": PSGC_FOLDER})
print(f"✓ STARTED record inserted for {run_id}")

In [0]:
folder_listing = spark.sql(f"""
SELECT path, file, size, modification_time
FROM list_files('{PSGC_FOLDER}', connection => '{GOOGLE_DRIVE_CONNECTION}')
ORDER BY path
""")
print(f"Total items found: {folder_listing.count()}")
display(folder_listing)

from urllib.parse import unquote

def _filename(path_value):
    """The Drive connector returns the name inside the path, e.g. '...?mimeType=...&filename=NAME.xlsx'."""
    m = re.search(r"[?&]filename=([^&']+)", str(path_value))
    return unquote(m.group(1)) if m else None

name_to_path = {}
for r in folder_listing.select("path", "file").collect():
    file_val = r["file"]
    uri = None
    try:
        uri = file_val["uri"]          # struct/Row returned by list_files
    except Exception:
        uri = getattr(file_val, "uri", None)
    name = _filename(r["path"]) or _filename(uri) or _filename(file_val)
    if name:
        name_to_path[name] = r["path"]

# Fill each dataset's load path from the listing (unless a url was set manually)
missing = []
for ds in datasets:
    src = SOURCE_FILES[ds]
    if src["url"] is None:
        src["url"] = name_to_path.get(src["file"])
    if src["url"] is None:
        missing.append(src["file"])
for ds in datasets:
    print(f"  {ds}: {SOURCE_FILES[ds]['file']} -> {SOURCE_FILES[ds]['url']}")
assert not missing, f"These files were not found in PSGC_FOLDER: {missing}. Check the folder URL, the filenames, or set \"url\" manually in SOURCE_FILES."

## Ingest Each Excel Workbook

For each selected dataset: read the workbook bytes from Drive, extract every sheet, run checks, and write one Bronze table per sheet.

If a **critical check fails** for a sheet, nothing is written for that sheet. It is recorded as `NEEDS_REVIEW` in `file_manifest` and the reason goes to `ingestion_errors`. The other sheets of the same workbook still load.

In [0]:
for ds in datasets:
    src = SOURCE_FILES[ds]
    url, fname = src["url"], src["file"]
    print(f"\n{'='*60}\n{ds} — {fname}")
    try:
        binary_df = (spark.read.format("binaryFile")
                     .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
                     .load(url))
        xlsx_bytes = binary_df.collect()[0]["content"]
        sheets = parse_excel(xlsx_bytes, ds)
        print(f"  Sheets found: {[s['sheet'] for s in sheets]}")
    except Exception as e:
        print(f"✗ FAILED: {ds} — {e}")
        log_ingestion_error(ds, fname, None, url, None, "INGESTION_ERROR", str(e))
        record_ingestion(ds, fname, None, f"{src['table_prefix']}_*", 0, 0, "FAILED", str(e), url)
        continue

    sheet_ok = 0
    for sh in sheets:
        tbl = tbl_sheet(ds, sh["sheet"])
        try:
            errors, notes = check_sheet(sh)
            if errors:
                msg = "; ".join(errors)
                print(f"  ✗ NEEDS_REVIEW [{sh['sheet']}]: {msg}")
                log_ingestion_error(ds, fname, sh["sheet"], url, tbl, "EXTRACTION_CHECK_FAILED", msg)
                record_ingestion(ds, fname, sh["sheet"], tbl, 0, 0, "NEEDS_REVIEW", msg, url)
                continue

            schema = ", ".join(f"`{c}` STRING" for c in sh["columns"])
            sheet_df = add_bronze_metadata(spark.createDataFrame(sh["rows"], schema=schema), ds, fname, sh["sheet"], url)
            write_table(sheet_df, tbl)

            n = spark.table(bronze_table(tbl)).count()
            if n != len(sh["rows"]):
                raise AssertionError(f"Row count mismatch in {tbl}: wrote {n}, read {len(sh['rows'])}")
            check_note = (f"header_row={sh['header_row']}; rows={n}; cols={len(sh['columns'])}; "
                          f"rows written = rows read" + (f"; {' | '.join(notes)}" if notes else ""))
            record_ingestion(ds, fname, sh["sheet"], tbl, n, len(sheet_df.columns), "INGESTED", check_note, url)
            sheet_ok += 1
            print(f"  ✓ INGESTED [{sh['sheet']}] -> {tbl}: {n} rows, {len(sh['columns'])} columns")
        except Exception as e:
            print(f"  ✗ FAILED [{sh['sheet']}]: {e}")
            log_ingestion_error(ds, fname, sh["sheet"], url, tbl, "INGESTION_ERROR", str(e))
            record_ingestion(ds, fname, sh["sheet"], tbl, 0, 0, "FAILED", str(e), url)

    # Original workbook as binary (traceability) — only if at least one sheet loaded
    if sheet_ok:
        try:
            files_df = add_bronze_metadata(
                binary_df.select(F.col("path"), F.col("length").alias("file_size"), F.col("content").alias("binary_content")),
                ds, fname, None, url)
            write_dataset_part(files_df, TBL_FILES, ds)
            n = spark.table(bronze_table(TBL_FILES)).filter(F.col("_source_dataset_id") == ds).count()
            record_ingestion(ds, fname, None, TBL_FILES, n, len(files_df.columns), "INGESTED", "original Excel file stored as binary", url)
            print(f"  ✓ INGESTED original file -> {TBL_FILES}")
        except Exception as e:
            print(f"  ✗ FAILED [original file]: {e}")
            log_ingestion_error(ds, fname, None, url, TBL_FILES, "INGESTION_ERROR", str(e))
            record_ingestion(ds, fname, None, TBL_FILES, 0, 0, "FAILED", str(e), url)

## Validate Bronze Tables

Read back every table written in this run and confirm metadata columns, a single source file, and row counts that match the manifest. Rerunning the same dataset should give the same counts (no duplicates).

In [0]:
checks = []

def _check(tbl, t, dataset_id, expect_sheet):
    for col in METADATA_COLS:
        assert col in t.columns, f"Missing metadata column {col} in {tbl}"
    assert t.filter(F.col("_source_system") != SOURCE_SYSTEM).count() == 0
    assert t.filter(F.col("_source_dataset_id") != dataset_id).count() == 0
    assert t.filter(F.col("_source_path").isNull() | F.col("_ingested_at").isNull()).count() == 0
    if expect_sheet:
        assert t.filter(F.col("_source_sheet").isNull()).count() == 0, f"_source_sheet is NULL in {tbl}"
        assert t.select("_source_sheet").distinct().count() == 1, f"More than one sheet in {tbl}"
    assert t.select("_source_file").distinct().count() == 1, f"More than one source file for {dataset_id} in {tbl}"

ingested = [r for r in ingestion_summary if r["status"] == "INGESTED"]
for r in ingested:
    ds, tbl = r["source_dataset_id"], r["bronze_table"]
    t = spark.table(bronze_table(tbl))
    is_sheet_table = r["source_sheet"] is not None
    if not is_sheet_table:
        t = t.filter(F.col("_source_dataset_id") == ds)
    _check(tbl, t, ds, is_sheet_table)
    n = t.count()
    checks.append((ds, r["source_sheet"], tbl, n, "ok" if n == r["row_count"] else f"count differs from manifest ({r['row_count']})"))

for r in ingestion_summary:
    if r["status"] != "INGESTED":
        checks.append((r["source_dataset_id"], r["source_sheet"], r["bronze_table"], None, r["status"]))

display(spark.createDataFrame(checks, "dataset STRING, sheet STRING, table STRING, row_count BIGINT, result STRING"))

## Ingestion Summary and Run Log Update

Updates this run's `ingestion_run_log` row from `STARTED` to `SUCCESS`, `PARTIAL_SUCCESS`, or `FAILED`.

In [0]:
manifest_df = spark.table(ingestion_table("file_manifest")).filter(F.col("run_id") == run_id)
manifest_df.createOrReplaceTempView("_run_manifest")
spark.sql("""
SELECT source_file,
  CASE WHEN SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) > 0 THEN 'FAILED'
       WHEN SUM(CASE WHEN status = 'NEEDS_REVIEW' THEN 1 ELSE 0 END) > 0 THEN 'SKIPPED'
       WHEN SUM(CASE WHEN status = 'INGESTED' THEN 1 ELSE 0 END) > 0 THEN 'INGESTED'
       ELSE 'SKIPPED' END AS file_status
FROM _run_manifest GROUP BY source_file
""").createOrReplaceTempView("_file_rollup")

files_discovered = spark.sql("SELECT COUNT(*) FROM _file_rollup").collect()[0][0]
files_ingested   = spark.sql("SELECT COUNT(*) FROM _file_rollup WHERE file_status='INGESTED'").collect()[0][0]
files_skipped    = spark.sql("SELECT COUNT(*) FROM _file_rollup WHERE file_status='SKIPPED'").collect()[0][0]
files_failed     = spark.sql("SELECT COUNT(*) FROM _file_rollup WHERE file_status='FAILED'").collect()[0][0]
tables_created   = spark.sql("SELECT COUNT(DISTINCT destination_table) FROM _run_manifest WHERE status='INGESTED'").collect()[0][0]
error_count      = spark.table(ingestion_table("ingestion_errors")).filter(F.col("run_id") == run_id).count()

if files_failed == 0 and files_skipped == 0 and error_count == 0:
    final_status = "SUCCESS"
elif files_ingested > 0 or tables_created > 0:
    final_status = "PARTIAL_SUCCESS"
else:
    final_status = "FAILED"
final_notes = (f"PSGC ingestion: {files_discovered} files processed, {files_ingested} ingested, "
               f"{files_skipped} needs review, {files_failed} failed, {tables_created} Bronze tables, {error_count} errors.")

spark.sql(f"""
UPDATE {ingestion_table('ingestion_run_log')}
SET status = :status, completed_at = current_timestamp(),
    files_discovered = :fd, files_ingested = :fi, files_skipped = :fs, files_failed = :ff,
    tables_created = :tc, notes = :notes
WHERE run_id = :run_id
""", args={"status": final_status, "fd": files_discovered, "fi": files_ingested, "fs": files_skipped,
           "ff": files_failed, "tc": tables_created, "notes": final_notes, "run_id": run_id})

print(f"Run {run_id}: {final_status}\n{final_notes}")
display(spark.table(ingestion_table("ingestion_run_log")).filter(F.col("run_id") == run_id))
display(manifest_df.orderBy("source_file", "source_sheet", "destination_table"))
display(spark.table(ingestion_table("ingestion_errors")).filter(F.col("run_id") == run_id))